<a href="https://colab.research.google.com/github/w-b-d/Basketball-Action-Detector/blob/main/basketball_pipeline_mvp_v5_botsort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basketball Pipeline — MVP v5 (BoT-SORT)

Detect → track → identify → project → analyze, tuned for occlusion robustness.

## What changed in v5 and why

The screen-occlusion losses traced to two things: the detector's confidence floor discarding
weak boxes during occlusion, and supervision's ByteTrack having no camera-motion compensation.
Both are addressed by switching the tracker.

**Tracker: `sv.ByteTrack` → `BoTSORTTracker`** (Roboflow `trackers` library). On **SportsMOT**
— the benchmark for "sports broadcast tracking with fast motion, camera pans, and similar-looking
targets," i.e. exactly this problem — the published comparison puts BoT-SORT first:

| Tracker | HOTA (default) | HOTA (tuned) |
|---|---|---|
| SORT | 70.8 | 72.9 |
| ByteTrack *(what v4 used)* | 73.0 | 73.3 |
| OC-SORT | 71.7 | 74.0 |
| **BoT-SORT** | **73.8** | **74.1** |
| C-BIoU | 73.1 | 73.1 |

BoT-SORT keeps ByteTrack's two-stage association (high-confidence first, then low-confidence
recovery for unmatched tracks — which is what rescues a player mid-occlusion) and adds **camera
motion compensation**: it estimates global camera motion between frames and warps each track's
predicted box before matching, so a pan doesn't make players appear to jump. It also blends
detection confidence into the association cost.

The tracker config below is Roboflow's **grid-searched SportsMOT configuration**, not a guess.

**Detector confidence 0.4 → 0.25.** Two-stage association only works if weak boxes reach the
tracker. At 0.4 the occlusion-time detections were being discarded before BoT-SORT saw them.
The tracker's own thresholds now do the filtering.

**Team classifier → HSV colour clustering.** Brightness thresholding cannot separate teams that
differ in hue but not lightness (Warriors blue vs Clippers red) — that was the Podziemski-labelled-
Kawhi failure. HSV clustering separates by hue and costs the same.

**Roster decides team (FIX 8).** A confirmed jersey number unique to one roster now sets the
team directly and anchors the vote, instead of trusting a possibly-wrong appearance vote.

Carried over from v4: duplicate-box dedup, on-court filter, repaired OCR call with loud errors,
number size gate, strided team vote, honest labels, number-based track stitching.

**Run:** top to bottom. After the install cell → **Runtime → Restart session → Run all.**


## 0. Setup

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    # fallback for non-Colab environments
    os.environ.setdefault("ROBOFLOW_API_KEY", input("ROBOFLOW_API_KEY: "))
    os.environ.setdefault("HF_TOKEN", input("HF_TOKEN: "))


In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
HOME = Path.cwd()
print("HOME:", HOME)

In [ ]:
# Guarded install — skips entirely when packages are already on this VM.
import importlib.util

if importlib.util.find_spec("trackers") is None:
    print("Fresh VM — installing (a few minutes)")
    !pip install -q "pillow>=12.3,<13" gdown pyarrow inference-gpu supervision==0.27.0 trackers pyDeprecate transformers num2words git+https://github.com/roboflow/sports.git@feat/basketball
    !pip install -q --force-reinstall --no-deps "pillow==12.3.0"
    print(">>> Now do: Runtime -> Restart session, then Run all. <<<")
else:
    print("Packages already present — continuing.")

In [ ]:
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

### Sanity check — numpy / supervision / trackers all import together

Adding a package can disturb the stack (this bit us with numpy 2.x before). Verify before a long run.

In [ ]:
import numpy as np, supervision as sv, trackers
print("numpy      :", np.__version__)
print("supervision:", sv.__version__)
print("trackers   :", getattr(trackers, "__version__", "unknown"))
from trackers import BoTSORTTracker
print("BoTSORTTracker import OK")

## 1. Source clip

720p / 30fps is the sweet spot. Downscale first: `ffmpeg -i in.mp4 -vf scale=-2:720,fps=30 out.mp4`

In [ ]:
SOURCE_VIDEO_DIRECTORY = HOME / "source"

if not SOURCE_VIDEO_DIRECTORY.exists():
    !gdown -q https://drive.google.com/drive/folders/1Ap-wGeV8DhgOwFNs5vydcAqhCOpVqBc2  -O {SOURCE_VIDEO_DIRECTORY} --folder

!ls -la {SOURCE_VIDEO_DIRECTORY}

SOURCE_VIDEO_PATH = SOURCE_VIDEO_DIRECTORY / "warriors_clippers_45sec_test.mp4"


## 2. Imports, models, constants

In [ ]:
from typing import Dict, List, Optional
from collections import defaultdict, Counter
import json

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from scipy.signal import savgol_filter
from IPython.display import Video

import supervision as sv
from inference import get_model
from sports import TeamClassifier, MeasurementUnit, ViewTransformer
from sports.basketball import (
    CourtConfiguration,
    League,
    draw_court,
    draw_points_on_court,
    draw_paths_on_court,
    ShotEventTracker,
)


In [ ]:
# ---- models -----------------------------------------------------------
PLAYER_DETECTION_MODEL_ID = "basketball-player-detection-3-ycjdo/4"
# LOWERED 0.4 -> 0.25: BoT-SORT's second association pass needs weak boxes to
# exist during occlusion. The tracker's thresholds do the filtering now.
PLAYER_DETECTION_MODEL_CONFIDENCE = 0.25
PLAYER_DETECTION_MODEL_IOU_THRESHOLD = 0.9
PLAYER_DETECTION_MODEL = get_model(model_id=PLAYER_DETECTION_MODEL_ID)

KEYPOINT_DETECTION_MODEL_ID = "basketball-court-detection-2/14"
KEYPOINT_DETECTION_MODEL_CONFIDENCE = 0.3
KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE = 0.5
KEYPOINT_DETECTION_MODEL = get_model(model_id=KEYPOINT_DETECTION_MODEL_ID)

NUMBER_RECOGNITION_MODEL_ID = "basketball-jersey-numbers-ocr/3"
NUMBER_RECOGNITION_MODEL = get_model(model_id=NUMBER_RECOGNITION_MODEL_ID)
NUMBER_RECOGNITION_MODEL_PROMPT = "Read the number."

# ---- detection classes ------------------------------------------------
# 0 ball | 1 ball-in-basket | 2 number | 3 player | 4 player-in-possession
# 5 player-jump-shot | 6 player-layup-dunk | 7 player-shot-block | 8 referee | 9 rim
NUMBER_CLASS_ID = 2
BALL_IN_BASKET_CLASS_ID = 1
PLAYER_CLASS_IDS = [3, 4, 5, 6, 7]
IN_POSSESSION_CLASS_ID = 4
JUMP_SHOT_CLASS_ID = 5
LAYUP_DUNK_CLASS_ID = 6

# ---- court ------------------------------------------------------------
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
FPS = video_info.fps
DT = 1.0 / FPS
print(video_info)

# ---- pipeline tuning --------------------------------------------------
IOU_DEDUP = 0.9          # merge player / player-in-possession duplicate boxes
COURT_MARGIN_FT = 5.0    # keep players this far past the lines
TEAM_VOTE_STRIDE = 3     # team vote every N frames (votes accumulate)
KEYPOINT_STRIDE = 5      # court keypoints every N frames (homography held between)
OCR_STRIDE = 15          # jersey OCR every N frames
NUMBER_MIN_PX = 12       # skip number boxes smaller than this
N_CONSECUTIVE = 2        # agreeing OCR reads needed to confirm a number
IDENTITY_TEAM_ANCHOR = 100   # team votes added when a roster identity confirms

# ---- rosters: EDIT for your clip --------------------------------------
ROSTERS = {
"Clippers" : {
    0: "Bradley Beal",
    00: "Sean Pedulla",
    2: "Kawhi Leonard",
    4: "Kobe Sanders",
    5: "Derrick Jones Jr.",
    7: "Bogdan Bogdanović",
    8: "Kris Dunn",
    9: "Bennedict Mathurin",
    10: "Darius Garland",
    11: "Brook Lopez",
    12: "Cam Christie",
    14: "TyTy Washington Jr.",
    20: "John Collins",
    22: "Jordan Miller",
    23: "Isaiah Jackson",
    33: "Nicolas Batum",
    55: "Norchad Omier",
},

"Warriors" : {
    0: "Gary Payton II",
    2: "Brandin Podziemski",
    3: "Will Richard",
    4: "Moses Moody",
    7: "Kristaps Porziņģis",
    8: "De'Anthony Melton",
    10: "Jimmy Butler III",
    15: "Gui Santos",
    18: "LJ Cryer",
    19: "Nate Williams",
    20: "Al Horford",
    21: "Quinten Post",
    23: "Draymond Green",
    28: "Charles Bassey",
    30: "Stephen Curry",
    31: "Seth Curry",
    33: "Malevy Leons",
    61: "Pat Spencer",
},
}

### Sanity check — single-frame detection

In [ ]:
COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])
box_annotator = sv.BoxAnnotator(color=COLOR, thickness=2)
label_annotator = sv.LabelAnnotator(color=COLOR, text_color=sv.Color.BLACK)

frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
result = PLAYER_DETECTION_MODEL.infer(
    frame,
    confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
    iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD,
)[0]
detections = sv.Detections.from_inference(result)

annotated = box_annotator.annotate(frame.copy(), detections)
annotated = label_annotator.annotate(annotated, detections)
sv.plot_image(annotated)


## 3. Team classifier (HSV colour clustering)

Clusters on mean HSV so teams separate by **hue**, not just lightness. Brightness-only
clustering fails when both jerseys are similarly dark/light but different colours.

Cluster ids are arbitrary — **eyeball the two grids and set `CLUSTER_TO_TEAM`.**

In [ ]:
from sklearn.cluster import KMeans

class ColorTeamClassifier:
    """KMeans on mean HSV of the crop. Separates by hue (blue vs red), which
    brightness thresholding cannot. Same .fit()/.predict() interface."""
    def __init__(self):
        self.km = None
    def _feat(self, crop):
        if crop is None or crop.size == 0:
            return np.zeros(3)
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        return hsv.reshape(-1, 3).mean(axis=0)
    def fit(self, crops):
        X = np.array([self._feat(c) for c in crops])
        self.km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
        print("cluster centers (H,S,V):", self.km.cluster_centers_.round(1))
    def predict(self, crops):
        X = np.array([self._feat(c) for c in crops])
        return self.km.predict(X).astype(int)


STRIDE = 30
crops = []
for f in tqdm(sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=STRIDE)):
    result = PLAYER_DETECTION_MODEL.infer(
        f, confidence=0.4,                    # higher bar for CLEAN training crops
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD,
        class_agnostic_nms=True)[0]
    dets = sv.Detections.from_inference(result)
    dets = dets[np.isin(dets.class_id, PLAYER_CLASS_IDS)]
    for box in sv.scale_boxes(xyxy=dets.xyxy, factor=0.4):
        c = sv.crop_image(f, box)
        if c is None or c.size == 0:
            continue
        h, w = c.shape[:2]
        if h == 0 or w == 0 or w * h < 400 or (w / h) < 0.35:   # drop slivers
            continue
        crops.append(c)

print(f"collected {len(crops)} crops")
team_classifier = ColorTeamClassifier()
team_classifier.fit(crops)

teams = team_classifier.predict(crops)
print("cluster 0:"); sv.plot_images_grid(
    [c for c, t in zip(crops, teams) if t == 0][:20], grid_size=(2, 10), size=(10, 2))
print("cluster 1:"); sv.plot_images_grid(
    [c for c, t in zip(crops, teams) if t == 1][:20], grid_size=(2, 10), size=(10, 2))

**Set the mapping** from the two grids above. Names must match `ROSTERS` keys exactly.

In [ ]:
CLUSTER_TO_TEAM = {0: "Clippers", 1: "Warriors"}     # EDIT if the grids are swapped
TEAM_TO_CLUSTER = {v: k for k, v in CLUSTER_TO_TEAM.items()}
assert set(CLUSTER_TO_TEAM.values()) == set(ROSTERS.keys()), \
    "CLUSTER_TO_TEAM names must match ROSTERS keys"
print(CLUSTER_TO_TEAM)

## 3.5 Identity layer & filters

In [ ]:
# ---------- IoS matching: number box inside player box ----------------------
def ios_batch(player_xyxy: np.ndarray, number_xyxy: np.ndarray) -> np.ndarray:
    """Intersection over Smaller area; rows=players, cols=numbers."""
    if len(player_xyxy) == 0 or len(number_xyxy) == 0:
        return np.zeros((len(player_xyxy), len(number_xyxy)))
    px1, py1, px2, py2 = [player_xyxy[:, i][:, None] for i in range(4)]
    nx1, ny1, nx2, ny2 = [number_xyxy[:, i][None, :] for i in range(4)]
    ix = np.clip(np.minimum(px2, nx2) - np.maximum(px1, nx1), 0, None)
    iy = np.clip(np.minimum(py2, ny2) - np.maximum(py1, ny1), 0, None)
    n_area = np.clip((nx2 - nx1) * (ny2 - ny1), 1e-6, None)
    return (ix * iy) / n_area


class ConsecutiveValidator:
    def __init__(self, n_consecutive: int = 3):
        self.n = n_consecutive
        self.streak: Dict[int, tuple] = {}
        self.confirmed: Dict[int, str] = {}

    def update(self, track_id: int, value: str) -> Optional[str]:
        value = "".join(ch for ch in str(value) if ch.isdigit())
        if not value:
            return None
        last_val, count = self.streak.get(track_id, (None, 0))
        count = count + 1 if value == last_val else 1
        self.streak[track_id] = (value, count)
        if count >= self.n:
            self.confirmed[track_id] = value
            return value
        return None


class PlayerRegistry:
    def __init__(self, rosters: Dict[str, Dict[int, str]]):
        self.rosters = rosters
        self.track_to_player: Dict[int, str] = {}
        self.player_to_track: Dict[str, int] = {}
        self.log: List[dict] = []

    def confirm(self, frame: int, track_id: int, team_name: str, number: str):
        name = self.rosters.get(team_name, {}).get(int(number))
        if name is None:
            return
        prev = self.track_to_player.get(track_id)
        if prev is not None and prev != name:
            self.player_to_track.pop(prev, None)
        old_track = self.player_to_track.get(name)
        if old_track is not None and old_track != track_id:
            self.track_to_player.pop(old_track, None)
        self.track_to_player[track_id] = name
        self.player_to_track[name] = track_id
        self.log.append({"frame": frame, "track_id": track_id,
                         "team": team_name, "number": number, "player": name})

    def name_of(self, track_id: int) -> Optional[str]:
        return self.track_to_player.get(track_id)


def dedup_players(det: sv.Detections, iou_thresh: float = IOU_DEDUP) -> sv.Detections:
    """Greedy confidence-ordered NMS across classes: merges the player /
    player-in-possession double detection of the same person."""
    if len(det) <= 1:
        return det
    order = (np.argsort(-det.confidence) if det.confidence is not None
             else np.arange(len(det)))
    keep, kept = [], []
    for i in order:
        b = det.xyxy[i]
        dup = False
        for kb in kept:
            xx1 = max(b[0], kb[0]); yy1 = max(b[1], kb[1])
            xx2 = min(b[2], kb[2]); yy2 = min(b[3], kb[3])
            inter = max(0.0, xx2 - xx1) * max(0.0, yy2 - yy1)
            union = ((b[2]-b[0])*(b[3]-b[1]) + (kb[2]-kb[0])*(kb[3]-kb[1]) - inter)
            if union > 0 and inter / union > iou_thresh:
                dup = True; break
        if not dup:
            keep.append(i); kept.append(b)
    return det[np.array(sorted(keep))]


_court_v = np.array(config.vertices, dtype=float)
COURT_XMIN, COURT_YMIN = _court_v.min(axis=0) - COURT_MARGIN_FT
COURT_XMAX, COURT_YMAX = _court_v.max(axis=0) + COURT_MARGIN_FT

def filter_on_court(det: sv.Detections,
                    transformer: Optional[ViewTransformer]) -> sv.Detections:
    """Drop detections whose foot point maps outside the court (+margin).
    Uses the previous frame's homography; passes all through until first fix."""
    if transformer is None or len(det) == 0:
        return det
    feet = det.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
    xy = transformer.transform_points(points=feet)
    keep = ((xy[:, 0] >= COURT_XMIN) & (xy[:, 0] <= COURT_XMAX) &
            (xy[:, 1] >= COURT_YMIN) & (xy[:, 1] <= COURT_YMAX))
    return det[keep]

## 3.6 BoT-SORT tracker

Parameters are Roboflow's **grid-searched SportsMOT configuration**. Note it was tuned with
YOLOX detections; RF-DETR's confidence distribution differs, so if too few tracks start, use
the permissive fallback in the notes at the bottom.

`enable_cmc=True` is the camera-motion compensation — keep it on for broadcast footage.

In [ ]:
from trackers import BoTSORTTracker

# Roboflow grid-searched SportsMOT config
BOTSORT_KWARGS = dict(
    lost_track_buffer=30,
    minimum_consecutive_frames=2,
    minimum_iou_threshold_first_assoc=0.1,
    minimum_iou_threshold_second_assoc=0.5,
    minimum_iou_threshold_unconfirmed_assoc=0.3,
    high_conf_det_threshold=0.7,
    track_activation_threshold=0.8,
    enable_cmc=True,
)

def make_tracker():
    """Construct BoT-SORT, dropping any kwarg this version doesn't accept."""
    kw = dict(BOTSORT_KWARGS)
    while True:
        try:
            return BoTSORTTracker(**kw)
        except TypeError as e:
            msg = str(e)
            dropped = None
            for k in list(kw):
                if k in msg:
                    kw.pop(k); dropped = k; break
            if dropped is None:
                raise
            print(f"  note: this trackers version rejected '{dropped}' — dropped it")

tracker = make_tracker()
print("tracker:", type(tracker).__name__, "| params:", BOTSORT_KWARGS)

## 4. Main loop

Per frame: detect → dedup → on-court filter → **BoT-SORT (with frame for CMC)** → team vote →
OCR → calibrate → project → annotate.

In [ ]:
tracker = make_tracker()          # fresh tracker state

team_votes: Dict[int, Counter] = defaultdict(Counter)
records: List[dict] = []
shot_flags: List[dict] = []

number_validator = ConsecutiveValidator(n_consecutive=N_CONSECUTIVE)
registry = PlayerRegistry(ROSTERS)

last_transformer: Optional[ViewTransformer] = None
frames_since_good_h = 10**9
MAX_H_HOLD_FRAMES = int(FPS * 2)

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(["#3399ff", "#ff6666"]), thickness=2)
name_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#3399ff", "#ff6666"]),
    text_color=sv.Color.WHITE, text_position=sv.Position.BOTTOM_CENTER)

ANNOTATED_VIDEO_PATH = HOME / f"{SOURCE_VIDEO_PATH.stem}-annotated.mp4"

with sv.VideoSink(ANNOTATED_VIDEO_PATH, video_info) as sink:
    for frame_idx, frame in tqdm(
        enumerate(sv.get_video_frames_generator(SOURCE_VIDEO_PATH)),
        total=video_info.total_frames,
    ):
        frame_h, frame_w, *_ = frame.shape
        result = PLAYER_DETECTION_MODEL.infer(
            frame,
            confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
            iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD,
        )[0]
        detections = sv.Detections.from_inference(result)

        shot_flags.append({
            "frame": frame_idx,
            "has_jump_shot": bool((detections.class_id == JUMP_SHOT_CLASS_ID).any()),
            "has_layup_dunk": bool((detections.class_id == LAYUP_DUNK_CLASS_ID).any()),
            "has_ball_in_basket": bool((detections.class_id == BALL_IN_BASKET_CLASS_ID).any()),
        })

        # ---- players: classes -> dedup -> on-court -> BoT-SORT --------------
        players = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]
        players = dedup_players(players)
        players = filter_on_court(players, last_transformer)
        # frame= enables camera motion compensation
        players = tracker.update(players, frame=frame)

        if players.tracker_id is None or len(players) == 0:
            sink.write_frame(frame)
            frames_since_good_h += 1
            continue

        # ---- team vote (strided, guarded) ----------------------------------
        if frame_idx % TEAM_VOTE_STRIDE == 0:
            boxes = sv.scale_boxes(xyxy=players.xyxy, factor=0.4)
            idx_crops = [(i, sv.crop_image(frame, b)) for i, b in enumerate(boxes)]
            idx_crops = [(i, c) for i, c in idx_crops if c is not None and c.size > 0]
            if idx_crops:
                preds = team_classifier.predict([c for _, c in idx_crops])
                for (i, _), p in zip(idx_crops, preds):
                    team_votes[int(players.tracker_id[i])][int(p)] += 1
        team_ids = np.array([
            team_votes[int(tid)].most_common(1)[0][0] if team_votes[int(tid)] else 0
            for tid in players.tracker_id])

        # ---- jersey OCR ------------------------------------------------------
        if frame_idx % OCR_STRIDE == 0:
            numbers_det = detections[detections.class_id == NUMBER_CLASS_ID]
            if len(numbers_det) > 0:
                ios = ios_batch(players.xyxy, numbers_det.xyxy)
                for n_idx in range(len(numbers_det)):
                    p_idx = int(np.argmax(ios[:, n_idx]))
                    if ios[p_idx, n_idx] < 0.9:
                        continue
                    nb_box = numbers_det.xyxy[n_idx]
                    if min(nb_box[2]-nb_box[0], nb_box[3]-nb_box[1]) < NUMBER_MIN_PX:
                        continue
                    padded = sv.clip_boxes(
                        sv.pad_boxes(xyxy=numbers_det.xyxy[[n_idx]], px=10, py=10),
                        (frame_w, frame_h))[0]
                    crop = sv.resize_image(
                        sv.crop_image(frame, padded), resolution_wh=(224, 224))
                    try:
                        inputs, metadata = NUMBER_RECOGNITION_MODEL.preprocess(
                            crop, prompt=NUMBER_RECOGNITION_MODEL_PROMPT)
                        prediction = NUMBER_RECOGNITION_MODEL.predict(inputs)
                        response = NUMBER_RECOGNITION_MODEL.postprocess(
                            prediction, metadata)[0]
                        read = str(response.response)
                    except Exception as e:
                        print(f"OCR failed at frame {frame_idx}: {e}")
                        continue

                    tid = int(players.tracker_id[p_idx])
                    confirmed = number_validator.update(tid, read)
                    if confirmed is not None:
                        # roster decides the team when the number is unique to one
                        hits = [t for t, r in ROSTERS.items() if int(confirmed) in r]
                        if len(hits) == 1:
                            team_name = hits[0]
                        elif len(hits) == 2:
                            team_name = CLUSTER_TO_TEAM[int(team_ids[p_idx])]
                        else:
                            team_name = None
                        if team_name is not None:
                            registry.confirm(frame_idx, tid, team_name, confirmed)
                            team_votes[tid][TEAM_TO_CLUSTER[team_name]] += IDENTITY_TEAM_ANCHOR

        # ---- calibration (strided) -------------------------------------------
        if frame_idx % KEYPOINT_STRIDE == 0:
            kp_result = KEYPOINT_DETECTION_MODEL.infer(
                frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
            key_points = sv.KeyPoints.from_inference(kp_result)
            lm = key_points.confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE
            if np.count_nonzero(lm) >= 4:
                last_transformer = ViewTransformer(
                    source=key_points[:, lm].xy[0],
                    target=np.array(config.vertices)[lm])
                frames_since_good_h = 0
            else:
                frames_since_good_h += 1
        else:
            frames_since_good_h += 1
        h_ok = last_transformer is not None and frames_since_good_h <= MAX_H_HOLD_FRAMES

        # ---- projection -------------------------------------------------------
        if h_ok:
            frame_xy = players.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
            court_xy = last_transformer.transform_points(points=frame_xy)
            for tid, team, cls, (x_ft, y_ft) in zip(
                players.tracker_id, team_ids, players.class_id, court_xy):
                records.append({
                    "frame": frame_idx, "t": frame_idx * DT,
                    "track_id": int(tid), "team": int(team),
                    "in_possession": bool(cls == IN_POSSESSION_CLASS_ID),
                    "x_ft": float(x_ft), "y_ft": float(y_ft),
                    "h_fresh": frames_since_good_h == 0,
                })

        # ---- annotate (honest labels) -----------------------------------------
        labels = []
        for t in players.tracker_id:
            tid = int(t)
            name = registry.name_of(tid)
            num = number_validator.confirmed.get(tid)
            if name and num:
                labels.append(f"#{num} {name}")
            elif num:
                labels.append(f"#{num}")
            else:
                labels.append(f"id{tid}")
        annotated = ellipse_annotator.annotate(
            frame.copy(), players, custom_color_lookup=team_ids)
        annotated = name_annotator.annotate(
            annotated, players, labels=labels, custom_color_lookup=team_ids)
        sink.write_frame(annotated)

raw = pd.DataFrame(records)
print(f"{len(raw)} points, {raw.track_id.nunique()} raw tracks, "
      f"{len(registry.log)} confirmed identity reads")
pd.DataFrame(registry.log).tail()

In [ ]:
COMPRESSED = HOME / f"{ANNOTATED_VIDEO_PATH.stem}-c.mp4"
!ffmpeg -y -loglevel error -i {ANNOTATED_VIDEO_PATH} -vcodec libx264 -crf 28 {COMPRESSED}
Video(COMPRESSED, embed=True, width=720)


## 4.5 Track stitching

Merges fragments of the same player: gap ≤ 3s, same team, physically reachable distance.
Names propagate across merged fragments.

In [ ]:
STITCH_MAX_GAP_S = 3.0
STITCH_SPEED_FT_S = 25.0
STITCH_SLACK_FT = 5.0

summ = raw.groupby("track_id").agg(
    start=("frame", "min"), end=("frame", "max"),
    team=("team", lambda s: s.mode().iloc[0]))
first_pos = raw.sort_values("frame").groupby("track_id")[["x_ft", "y_ft"]].first()
last_pos = raw.sort_values("frame").groupby("track_id")[["x_ft", "y_ft"]].last()

parent: Dict[int, int] = {}
def find(t):
    while parent.get(t, t) != t:
        t = parent[t]
    return t

used_births = set()
for dead_id, d in summ.sort_values("end").iterrows():
    best, best_score = None, -1.0
    for new_id, b in summ.sort_values("start").iterrows():
        if new_id == dead_id or new_id in used_births:
            continue
        gap = (b.start - d.end) * DT
        if gap <= 0 or gap > STITCH_MAX_GAP_S or b.team != d.team:
            continue
        dist = np.hypot(first_pos.loc[new_id].x_ft - last_pos.loc[dead_id].x_ft,
                        first_pos.loc[new_id].y_ft - last_pos.loc[dead_id].y_ft)
        max_dist = STITCH_SPEED_FT_S * gap + STITCH_SLACK_FT
        if dist > max_dist:
            continue
        score = 1 - dist / max_dist
        if score > best_score:
            best, best_score = new_id, score
    if best is not None:
        parent[int(best)] = find(int(dead_id))
        used_births.add(best)

raw["entity_id"] = raw.track_id.map(lambda t: find(int(t)))
print(f"stitched {raw.track_id.nunique() - raw.entity_id.nunique()} fragments; "
      f"{raw.entity_id.nunique()} entities remain")

entity_name: Dict[int, str] = {}
for row in sorted(registry.log, key=lambda r: r["frame"]):
    entity_name[find(int(row["track_id"]))] = row["player"]
raw["player"] = raw.entity_id.map(entity_name).fillna("")
raw["track_id"] = raw["entity_id"]
print(raw[raw.player != ""].groupby("player").frame.count().rename("frames_named"))

## 5. Clean trajectories

In [ ]:
MIN_TRACK_SECONDS = 1.0
MAX_SPEED_FT_S = 30.0          # anything faster than ~30 ft/s is a tracking glitch
SMOOTH_WINDOW = 9              # frames (odd)
SMOOTH_POLY = 2

def clean_track(df: pd.DataFrame) -> Optional[pd.DataFrame]:
    df = df.sort_values("frame")
    if len(df) < int(MIN_TRACK_SECONDS * FPS):
        return None

    # reindex to contiguous frames, interpolate gaps
    full = pd.DataFrame({"frame": np.arange(df.frame.min(), df.frame.max() + 1)})
    df = full.merge(df, on="frame", how="left")
    df["track_id"] = df["track_id"].ffill().bfill()
    df["team"] = df["team"].ffill().bfill()
    df["in_possession"] = df["in_possession"].fillna(False)
    if "player" in df.columns:
        df["player"] = df["player"].ffill().bfill().fillna("")
    df[["x_ft", "y_ft"]] = df[["x_ft", "y_ft"]].interpolate(limit_direction="both")

    # kill teleport jumps: mask speed outliers, re-interpolate
    v = np.hypot(df.x_ft.diff(), df.y_ft.diff()) / DT
    bad = v > MAX_SPEED_FT_S
    df.loc[bad, ["x_ft", "y_ft"]] = np.nan
    df[["x_ft", "y_ft"]] = df[["x_ft", "y_ft"]].interpolate(limit_direction="both")

    # smooth
    if len(df) >= SMOOTH_WINDOW:
        df["x_ft"] = savgol_filter(df.x_ft, SMOOTH_WINDOW, SMOOTH_POLY)
        df["y_ft"] = savgol_filter(df.y_ft, SMOOTH_WINDOW, SMOOTH_POLY)

    df["t"] = df["frame"] * DT
    vx = np.gradient(df.x_ft, DT)
    vy = np.gradient(df.y_ft, DT)
    df["speed_ft_s"] = np.hypot(vx, vy)
    df["heading_rad"] = np.arctan2(vy, vx)
    return df

cleaned = [c for _, g in raw.groupby("track_id") if (c := clean_track(g)) is not None]
traj = pd.concat(cleaned, ignore_index=True)
traj.to_parquet(HOME / "trajectories.parquet")
print(f"{traj.track_id.nunique()} tracks kept, {len(traj)} rows -> trajectories.parquet")


### Top-down court video

In [ ]:
court_img = draw_court(config=config)
court_h, court_w, _ = court_img.shape

court_video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
court_video_info.width = court_w
court_video_info.height = court_h

COURT_VIDEO_PATH = HOME / f"{SOURCE_VIDEO_PATH.stem}-court.mp4"
TEAM_HEX = ["#3399ff", "#ff6666"]

by_frame = {f: g for f, g in traj.groupby("frame")}

with sv.VideoSink(COURT_VIDEO_PATH, court_video_info) as sink:
    for frame_idx in tqdm(range(int(traj.frame.max()) + 1)):
        court = draw_court(config=config)
        g = by_frame.get(frame_idx)
        if g is not None:
            for team in (0, 1):
                xy = g.loc[g.team == team, ["x_ft", "y_ft"]].to_numpy()
                if len(xy):
                    court = draw_points_on_court(
                        config=config, xy=xy,
                        fill_color=sv.Color.from_hex(TEAM_HEX[team]),
                        court=court,
                    )
        sink.write_frame(court)

COURT_COMPRESSED = HOME / f"{COURT_VIDEO_PATH.stem}-c.mp4"
!ffmpeg -y -loglevel error -i {COURT_VIDEO_PATH} -vcodec libx264 -crf 28 {COURT_COMPRESSED}
Video(COURT_COMPRESSED, embed=True, width=720)


## 6. Composite video — annotated broadcast + court inset

Everything in one artifact: identity labels on the broadcast, projected positions in the corner.
Pure rendering, no inference, so re-running is fast.

In [ ]:
INSET_SCALE = 0.30
INSET_MARGIN = 12
INSET_CORNER = "top-right"      # "top-left" | "top-right"
TEAM_HEX = ["#3399ff", "#ff6666"]

court_base = draw_court(config=config)
court_h0, court_w0, _ = court_base.shape
inset_w = int(video_info.width * INSET_SCALE)
inset_h = int(court_h0 * (inset_w / court_w0))

by_frame = {f: g for f, g in traj.groupby("frame")}
COMPOSITE_PATH = HOME / f"{SOURCE_VIDEO_PATH.stem}-composite.mp4"

with sv.VideoSink(COMPOSITE_PATH, video_info) as sink:
    for frame_idx, frame in tqdm(
        enumerate(sv.get_video_frames_generator(ANNOTATED_VIDEO_PATH)),
        total=video_info.total_frames,
    ):
        court = draw_court(config=config)
        g = by_frame.get(frame_idx)
        if g is not None:
            for team in (0, 1):
                xy = g.loc[g.team == team, ["x_ft", "y_ft"]].to_numpy()
                if len(xy):
                    court = draw_points_on_court(
                        config=config, xy=xy,
                        fill_color=sv.Color.from_hex(TEAM_HEX[team]), court=court)
        inset = cv2.resize(court, (inset_w, inset_h))
        inset = cv2.copyMakeBorder(inset, 2, 2, 2, 2,
                                   cv2.BORDER_CONSTANT, value=(255, 255, 255))
        ih, iw, _ = inset.shape
        y0 = INSET_MARGIN
        x0 = (INSET_MARGIN if INSET_CORNER == "top-left"
              else video_info.width - iw - INSET_MARGIN)
        frame[y0:y0+ih, x0:x0+iw] = inset
        sink.write_frame(frame)

COMPOSITE_C = HOME / f"{COMPOSITE_PATH.stem}-c.mp4"
!ffmpeg -y -loglevel error -i {COMPOSITE_PATH} -vcodec libx264 -crf 26 {COMPOSITE_C}
Video(COMPOSITE_C, embed=True, width=900)

## 7. Shot events

In [ ]:
shot_event_tracker = ShotEventTracker(
    reset_time_frames=int(FPS * 1.7),
    minimum_frames_between_starts=int(FPS * 0.5),
    cooldown_frames_after_made=int(FPS * 0.5),
)

shot_events = []
for flags in shot_flags:
    events = shot_event_tracker.update(
        frame_index=flags["frame"],
        has_jump_shot=flags["has_jump_shot"],
        has_layup_dunk=flags["has_layup_dunk"],
        has_ball_in_basket=flags["has_ball_in_basket"],
    )
    if events:
        for e in (events if isinstance(events, (list, tuple)) else [events]):
            shot_events.append({"frame": flags["frame"], "t": flags["frame"] * DT, "event": str(e)})

print(json.dumps(shot_events, indent=2))


## 8. On-ball screen detector

In [ ]:
SCREENER_MAX_SPEED = 2.5        # ft/s
SCREENER_MIN_STILL_S = 0.5      # s
CONTACT_DIST_FT = 6.0           # ft
MIN_HEADING_CHANGE_DEG = 30.0
HEADING_WINDOW_S = 1.0          # s after closest approach
MIN_GAP_BETWEEN_SCREENS_S = 2.0

def wrap_angle_deg(a):
    return np.abs((a + 180.0) % 360.0 - 180.0)

def detect_on_ball_screens(traj: pd.DataFrame) -> List[dict]:
    frames = np.arange(int(traj.frame.min()), int(traj.frame.max()) + 1)
    piv_x = traj.pivot_table(index="frame", columns="track_id", values="x_ft").reindex(frames)
    piv_y = traj.pivot_table(index="frame", columns="track_id", values="y_ft").reindex(frames)
    piv_v = traj.pivot_table(index="frame", columns="track_id", values="speed_ft_s").reindex(frames)
    piv_h = traj.pivot_table(index="frame", columns="track_id", values="heading_rad").reindex(frames)
    team_of = traj.groupby("track_id").team.agg(lambda s: s.mode().iloc[0]).to_dict()

    # ball handler per frame: the track flagged in_possession (mode-filled per frame)
    poss = traj[traj.in_possession].groupby("frame").track_id.first()
    handler = poss.reindex(frames).ffill(limit=int(FPS * 0.5))  # hold handler briefly

    still_frames = int(SCREENER_MIN_STILL_S * FPS)
    heading_frames = int(HEADING_WINDOW_S * FPS)
    events, last_event_frame = [], -10**9

    for tid in piv_x.columns:
        # candidate windows where this track is quasi-stationary
        still = (piv_v[tid] < SCREENER_MAX_SPEED).fillna(False).to_numpy()
        run = np.convolve(still.astype(int), np.ones(still_frames, int), mode="same")
        stationary = run >= still_frames

        for i, f in enumerate(frames):
            h = handler.get(f)
            if pd.isna(h) or int(h) == int(tid):
                continue
            if team_of.get(int(h)) != team_of.get(int(tid)):
                continue
            if not stationary[i]:
                continue

            dx = piv_x.at[f, int(h)] - piv_x.at[f, tid]
            dy = piv_y.at[f, int(h)] - piv_y.at[f, tid]
            if np.isnan(dx) or np.hypot(dx, dy) > CONTACT_DIST_FT:
                continue

            # heading change of the handler after closest approach
            f_end = min(f + heading_frames, frames[-1])
            try:
                h0 = np.degrees(piv_h.at[f, int(h)])
                h1 = np.degrees(piv_h.at[f_end, int(h)])
            except KeyError:
                continue
            if np.isnan(h0) or np.isnan(h1):
                continue
            if wrap_angle_deg(h1 - h0) < MIN_HEADING_CHANGE_DEG:
                continue

            if f - last_event_frame < MIN_GAP_BETWEEN_SCREENS_S * FPS:
                continue
            last_event_frame = f
            events.append({
                "type": "on_ball_screen",
                "frame": int(f),
                "t": round(float(f * DT), 2),
                "screener_track": int(tid),
                "handler_track": int(h),
                "heading_change_deg": round(float(wrap_angle_deg(h1 - h0)), 1),
            })
    return sorted(events, key=lambda e: e["frame"])

screen_events = detect_on_ball_screens(traj)
name_of_entity = traj[traj.player != ""].groupby("track_id").player.first().to_dict() if "player" in traj.columns else {}
for e in screen_events:
    e["screener"] = name_of_entity.get(e["screener_track"], "")
    e["handler"] = name_of_entity.get(e["handler_track"], "")
print(json.dumps(screen_events, indent=2))


In [ ]:
all_events = {"shots": shot_events, "screens": screen_events}
with open(HOME / "events.json", "w") as f:
    json.dump(all_events, f, indent=2)

# visualize the handler + screener paths for the first detected screen
if screen_events:
    e = screen_events[0]
    w = int(FPS * 2)
    win = traj[(traj.frame >= e["frame"] - w) & (traj.frame <= e["frame"] + w)]
    court = draw_paths_on_court(
        config=config,
        paths=[win[win.track_id == e["handler_track"]][["x_ft", "y_ft"]].to_numpy()],
        color=sv.Color.GREEN,
    )
    court = draw_paths_on_court(
        config=config,
        paths=[win[win.track_id == e["screener_track"]][["x_ft", "y_ft"]].to_numpy()],
        color=sv.Color.RED,
        court=court,
    )
    sv.plot_image(court)
print("saved -> events.json | trajectories.parquet | annotated + court videos")


## Tuning notes

**Too few tracks / players not picked up.** The SportsMOT config was tuned on YOLOX detections;
RF-DETR scores differently. Try the permissive variant:

```python
BOTSORT_KWARGS = dict(
    lost_track_buffer=60,                        # longer occlusion tolerance
    minimum_consecutive_frames=2,
    minimum_iou_threshold_first_assoc=0.1,
    minimum_iou_threshold_second_assoc=0.5,
    minimum_iou_threshold_unconfirmed_assoc=0.2,
    high_conf_det_threshold=0.5,                 # more detections in stage 1
    track_activation_threshold=0.5,              # easier to start a track
    enable_cmc=True,
)
```

**Still losing players behind screens.** Raise `lost_track_buffer` to 60–90 and
`STITCH_MAX_GAP_S` to match. Beyond that, the ambiguity is real — two players occupying the
same pixels is genuinely under-determined from one camera, and number-stitching is the safety net.

**ID switches during pans** should be much rarer than v4 — that's what `enable_cmc=True` buys.
If they persist, confirm CMC is actually active (the `frame=` argument must be passed).

**Wrong team on a player.** Check `CLUSTER_TO_TEAM` against the grids first. Then run:
```python
for tid, ctr in sorted(team_votes.items()):
    print(tid, dict(ctr), registry.name_of(tid) or "")
```
A near-tie means the colour clustering can't separate these uniforms; a lopsided-but-wrong vote
means the mapping is inverted.

**Numbers shared across rosters** (both teams have a #2) fall back to the appearance vote —
the roster-decides rule only fires when a number is unique to one team.

**Measuring the MVP.** Hand-label ~50 frames (player, team, number) and score against them.
`registry.log` plus the composite video are your evidence; without a labelled set, "80% accuracy"
isn't a claim you can defend.
